# 第 3 章　金融数据 API 获取

> **T1 共三章**
> - 第 1 章：Python 环境配置
> - 第 2 章：工具链（Git / Marp / Quarto / Stata×Python）
> - **第 3 章（本章）**：金融数据 API 获取

::: {.callout-note}
## 本章要点

1. **AKShare**：免费获取 A 股、港股、期货、基金、宏观数据
2. **CSMAR Python API**：结构化获取上市公司财务、股价数据
3. **FRED API**：美联储经济数据（利率、GDP、通胀等）
4. **World Bank API**：跨国宏观经济数据
5. **yfinance**：国际股票和 ETF 数据（附大陆网络替代方案）
6. **数据存储**：CSV / Parquet / SQLite 的适用场景与选择

本章是**工具参考章**——不需要一次全部记住，
需要哪类数据时来查对应的节即可。
每节都有可以直接复制运行的完整代码。
:::


---

## 0　引言：数据从哪里来

金融研究需要的数据，按获取难度和成本大致分三层：

| 层级 | 数据源 | 特点 |
|------|--------|------|
| **免费开源** | AKShare、FRED、World Bank | 即装即用，适合入门和宏观数据 |
| **校园授权** | CSMAR、Wind | 数据质量最高，覆盖最全；本校已采购 |
| **商业付费** | Bloomberg、Refinitiv | 机构级，本课程不使用 |

本章的学习策略：**先跑通，再深入**。
每个 API 都先用一个最小可运行的例子验证能拿到数据，
然后再了解更多参数和数据表。

贯穿本章的分析任务是课程第一次个人作业：
**获取沪深 300 近三年日度收益率，同时获取同期的 FRED 联邦基金利率，**
**计算两者的滚动相关性，绘制双轴时序图。**


## 环境准备


In [ ]:
# ── 第 3 章　金融数据 API 获取　───────────────────────────────────────────
import os, warnings, time
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_RAW = 'data_raw'
OUTPUT   = 'output'
for d in [DATA_RAW, OUTPUT]:
    os.makedirs(d, exist_ok=True)

print('环境就绪 ✓')


---

## 1　AKShare：免费的中国金融数据库

AKShare 是目前最完整的免费中国金融数据 Python 库，
覆盖 A 股、港股、期货、基金、债券、宏观经济等数百个数据接口。

- 文档：[https://akshare.akfamily.xyz](https://akshare.akfamily.xyz)
- 安装：`pip install akshare`（已在第 1 章安装）

### 1.1　AKShare 使用模式

AKShare 的所有函数名都遵循 `stock_xxx`、`fund_xxx`、`macro_xxx` 等前缀约定，
查不到某类数据时，直接在文档首页搜索关键词即可找到对应函数名。

::: {.callout-warning}
## 请求频率限制

AKShare 的部分接口有请求频率限制（通常每分钟几十次）。
批量获取多只股票数据时，在循环里加 `time.sleep(0.5)` 避免被封。
大批量数据建议一次获取后保存到本地，不要每次运行都重新拉取。
:::


In [ ]:
# ── 1.2  沪深 300 指数日度行情 ───────────────────────────────────────────
import akshare as ak

# 获取沪深 300 日度行情（近三年）
# symbol='sh000300' 是沪深 300 的代码；adjust='' 表示不复权
hs300 = ak.stock_zh_index_daily(symbol='sh000300')

# ── 基本处理 ─────────────────────────────────────────────────────────────
hs300['date'] = pd.to_datetime(hs300['date'])
hs300 = hs300.sort_values('date').reset_index(drop=True)

# 只取近三年
cutoff = pd.Timestamp.today() - pd.DateOffset(years=3)
hs300  = hs300[hs300['date'] >= cutoff].copy()

# 计算日度收益率（对数收益率）
hs300['ret'] = np.log(hs300['close'] / hs300['close'].shift(1))
hs300 = hs300.dropna(subset=['ret'])

print(f'沪深 300 日度数据：{len(hs300)} 行')
print(f'时间范围：{hs300["date"].min().date()} ~ {hs300["date"].max().date()}')
print(f'列名：{list(hs300.columns)}')
hs300.tail()


In [ ]:
# ── 1.3  保存原始数据 ────────────────────────────────────────────────────
# 养成习惯：拉到数据后立刻保存，之后的分析从本地读，不重复调用 API
save_path = f'{DATA_RAW}/hs300_daily.csv'
hs300.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f'已保存至 {save_path}  ({os.path.getsize(save_path)/1024:.1f} KB)')


In [ ]:
# ── 1.4  A 股个股历史行情（复权）────────────────────────────────────────
# 示例：平安银行（000001），前复权
stock = ak.stock_zh_a_hist(
    symbol='000001',
    period='daily',
    start_date='20220101',
    end_date='20241231',
    adjust='qfq'         # qfq=前复权，hfq=后复权，''=不复权
)

stock = stock.rename(columns={
    '日期': 'date', '开盘': 'open', '收盘': 'close',
    '最高': 'high', '最低': 'low', '成交量': 'volume',
    '成交额': 'amount', '涨跌幅': 'pct_chg'
})
stock['date'] = pd.to_datetime(stock['date'])

print(f'平安银行日度数据：{len(stock)} 行')
stock[['date','close','pct_chg','volume']].tail()


In [ ]:
# ── 1.5  宏观经济数据：CPI、PMI ──────────────────────────────────────────
# CPI 月度同比
cpi = ak.macro_china_cpi_monthly()
cpi.columns = ['date', 'cpi_yoy']
cpi['date'] = pd.to_datetime(cpi['date'])
cpi = cpi.sort_values('date').tail(36)   # 近 3 年

print('CPI 月度同比（近 3 年）：')
print(cpi.tail(6).to_string(index=False))


::: {.callout-tip}
## 提示词示范（AKShare 接口查询）

````
我需要用 AKShare 获取以下数据，请帮我找到对应的函数名和参数：
1. 沪深 300 成分股列表（当前最新）
2. 某只股票的分红派息历史记录
3. 全市场股票的实时市盈率（PE）数据

对每个函数，请给出：
- 函数名
- 主要参数说明
- 一个可运行的最小示例
- 返回 DataFrame 的主要列名说明
````
:::


---

## 2　CSMAR：中国上市公司专业数据库

CSMAR（国泰安）是中国最权威的上市公司研究数据库，
本校已采购授权，学生可免费使用。

CSMAR 提供两种获取方式：
1. **网页下载**：登录 [https://cn.gtadata.com](https://cn.gtadata.com)，
   手动选择数据表和字段，导出 CSV
2. **Python API**：程序化获取，适合大批量数据

### 2.1　API 配置

```bash
# 安装 CSMAR Python 包
pip install csmar
```

::: {.callout-note}
## API 密钥申请

CSMAR Python API 需要账号和密钥：
1. 用学校邮箱在 CSMAR 网站注册账号
2. 登录后在「个人中心 → API 密钥」处获取 `token`
3. 首次使用时运行 `csmar.login(token='你的密钥')` 完成认证

密钥不要写在代码里——放在环境变量或单独的 `config.py` 文件中，
把 `config.py` 加入 `.gitignore`。
:::


In [ ]:
# ── 2.2  CSMAR API 使用示例 ─────────────────────────────────────────────
# 注意：运行前需要先完成 2.1 节的 API 配置

try:
    import csmar
    CSMAR_AVAILABLE = True
    print('csmar 库已安装 ✓')
except ImportError:
    CSMAR_AVAILABLE = False
    print('csmar 库未安装，运行：pip install csmar')
    print('本节代码为演示性质，实际运行需先完成 API 配置。')

# ── 演示：获取上市公司基本信息 ───────────────────────────────────────────
# 以下代码在完成 API 配置后可以直接运行
if CSMAR_AVAILABLE:
    pass
    # ── 取消注释以下代码块来实际运行 ─────────────────────────────────────
    # csmar.login(token='你的API密钥')   # 或从环境变量读取
    #
    # # 上市公司基本信息（TRD_Co 表）
    # basicinf = csmar.get_table(
    #     table_name = 'TRD_Co',
    #     fields     = ['Stkcd', 'Stknme', 'Indcd', 'Listdt'],
    #     filters    = {'Listdt': ('>=', '2010-01-01')},
    # )
    # basicinf.to_csv(f'{DATA_RAW}/csmar_basicinf.csv', index=False)
    # print(basicinf.head())


### 2.3　CSMAR 常用数据表速查

| 数据类型 | 表名 | 主要字段 |
|----------|------|----------|
| 上市公司基本信息 | `TRD_Co` | `Stkcd`, `Stknme`, `Indcd`, `Listdt` |
| 股票日度行情 | `TRD_Dalyr` | `Stkcd`, `Trddt`, `Clsprc`, `Dretwd` |
| 财务报表（资产负债）| `FS_Combas` | `Stkcd`, `Accper`, `A001000000` |
| 财务比率 | `FI_T10` | `Stkcd`, `Accper`, `F100801A` (ROA) |
| 分析师预测 | `CSMAR_Analyst` | `Stkcd`, `Rptdt`, `Ftype`, `EPS` |
| 公司公告 | `CSMAR_Announce` | `Stkcd`, `AnnDt`, `DocType`, `DocUrl` |

> **完整表目录**：登录 CSMAR 网站后，在「数据产品 → 数据字典」里可以
> 搜索所有可用表名和字段。

::: {.callout-tip}
## 提示词示范（CSMAR 数据获取）

````
我需要用 CSMAR Python API 获取以下数据：
1. 2015–2023 年所有 A 股上市公司的年度合并报表数据，
   包含：股票代码、会计期间、总资产、净利润、营业收入、资产负债率
2. 只保留每年 12 月 31 日的数据（年报）

请帮我：
- 确认对应的 CSMAR 表名和字段名
- 写出完整的 API 调用代码
- 说明如何处理「合并报表 vs 母公司报表」的筛选
- 数据量约多大？建议如何存储（CSV / Parquet / SQLite）
````
:::


---

## 3　FRED API：美联储经济数据

FRED（Federal Reserve Economic Data）是美联储圣路易斯分行维护的
免费经济数据库，包含 80 万+ 个经济时序数据，
覆盖美国及全球的 GDP、利率、通胀、就业、汇率等指标。

- 文档：[https://fred.stlouisfed.org](https://fred.stlouisfed.org)
- 安装：`pip install fredapi`（已在第 1 章安装）
- **免费 API Key**：在 FRED 网站注册后获取（必须有 Key 才能使用 API）

### 3.1　获取 API Key

1. 访问 [https://fred.stlouisfed.org/docs/api/api_key.html](https://fred.stlouisfed.org/docs/api/api_key.html)
2. 注册账号（免费），在设置里申请 API Key
3. Key 是一串 32 位字符，保存好，不要上传到 GitHub

::: {.callout-warning}
## API Key 安全

永远不要把 API Key 直接写在代码里上传到 GitHub。
推荐做法：
```python
import os
FRED_KEY = os.environ.get('FRED_API_KEY', '')  # 从环境变量读取
# 或
from config import FRED_KEY                      # 从 .gitignore 里的 config.py 读取
```
:::


In [ ]:
# ── 3.2  FRED API 使用示例 ───────────────────────────────────────────────
from fredapi import Fred

# ── 替换为你的 API Key ───────────────────────────────────────────────────
# 推荐从环境变量读取：os.environ.get('FRED_API_KEY')
FRED_KEY = 'your_fred_api_key_here'   # ← 替换为你的真实 Key

if FRED_KEY == 'your_fred_api_key_here':
    print('请替换 FRED_KEY 为你的真实 API Key 后再运行。')
    print('申请地址：https://fred.stlouisfed.org/docs/api/api_key.html')
else:
    fred = Fred(api_key=FRED_KEY)

    # ── 联邦基金利率（月度）─────────────────────────────────────────────
    fedfunds = fred.get_series(
        'FEDFUNDS',
        observation_start='2022-01-01',
        observation_end='2024-12-31',
    )
    fedfunds = fedfunds.rename('fed_funds_rate').reset_index()
    fedfunds.columns = ['date', 'fed_funds_rate']

    print(f'联邦基金利率：{len(fedfunds)} 个月度数据')
    print(fedfunds.tail(6).to_string(index=False))
    fedfunds.to_csv(f'{DATA_RAW}/fred_fedfunds.csv', index=False)


In [ ]:
# ── 3.3  常用 FRED 数据系列速查 ──────────────────────────────────────────
# 以下代码打印常用系列 ID，供参考查阅

fred_series = {
    # 利率
    'FEDFUNDS'  : '联邦基金利率（月度，%）',
    'DGS10'     : '10 年期美国国债收益率（日度，%）',
    'DGS2'      : '2 年期美国国债收益率（日度，%）',
    'T10Y2Y'    : '10Y-2Y 利差（期限溢价，日度）',
    # 宏观经济
    'GDP'       : '美国 GDP（季度，十亿美元）',
    'CPIAUCSL'  : 'CPI（月度，1982-84=100）',
    'UNRATE'    : '美国失业率（月度，%）',
    'DEXCHUS'   : '美元/人民币汇率（日度）',
    # 金融市场
    'SP500'     : '标普 500 指数（日度）',
    'VIXCLS'    : 'VIX 波动率指数（日度）',
    'BAMLH0A0HYM2': '美国高收益债利差（日度，%）',
}

print(f'{'系列 ID':<16} 说明')
print('─' * 48)
for sid, desc in fred_series.items():
    print(f'{sid:<16} {desc}')

print()
print('完整搜索：https://fred.stlouisfed.org/  （右上角搜索框）')


::: {.callout-tip}
## 提示词示范（FRED 数据获取）

````
我需要用 fredapi 获取以下数据用于分析，请帮我：
1. 获取 2018–2024 年的月度美国 CPI 同比增速（CPIAUCSL）
2. 获取同期的联邦基金利率（FEDFUNDS）
3. 计算「泰勒规则缺口」= 实际利率 - 理论利率
   其中理论利率 = 2 + CPI同比 + 0.5 × (CPI同比 - 2) + 0.5 × 产出缺口
   产出缺口暂时用 0 代替
4. 绘制实际利率和泰勒规则利率的双线时序图
````
:::


---

## 4　World Bank API：跨国宏观数据

World Bank Open Data 提供 180+ 个国家的经济发展指标，
适合做跨国比较分析和国际政策研究。

- 安装：`pip install wbgapi`
- 无需 API Key，完全免费
- 指标搜索：[https://data.worldbank.org/indicator](https://data.worldbank.org/indicator)


In [ ]:
# ── 4.1  World Bank 数据获取示例 ─────────────────────────────────────────
import wbgapi as wb

# ── 常用指标 ID 速查 ─────────────────────────────────────────────────────
wb_indicators = {
    'NY.GDP.MKTP.CD'  : 'GDP（现价美元）',
    'NY.GDP.PCAP.CD'  : '人均 GDP（现价美元）',
    'FP.CPI.TOTL.ZG'  : 'CPI 通胀率（年度 %）',
    'FR.INR.LEND'     : '贷款利率（%）',
    'GC.DOD.TOTL.GD.ZS': '政府债务占 GDP 比（%）',
    'NE.TRD.GNFS.ZS'  : '贸易占 GDP 比（%）',
}

print(f'{'指标 ID':<30} 说明')
print('─' * 56)
for iid, desc in wb_indicators.items():
    print(f'{iid:<30} {desc}')


In [ ]:
# ── 4.2  获取金砖国家 GDP 数据 ───────────────────────────────────────────
brics = ['CN', 'IN', 'BR', 'RU', 'ZA']   # 中国、印度、巴西、俄罗斯、南非

gdp = wb.data.DataFrame(
    'NY.GDP.PCAP.CD',   # 人均 GDP（现价美元）
    brics,
    mrv=15,             # most recent values：取最近 15 年
    labels=True,        # 用国家名称替代代码
)

# wbgapi 返回的 DataFrame 行是国家，列是年份（需要转置为长表）
gdp_long = gdp.T.reset_index()
gdp_long.columns.name = None
gdp_long = gdp_long.rename(columns={'index': 'year'})
gdp_long['year'] = gdp_long['year'].astype(str).str.extract(r'(\d{4})').astype(int)

print(f'金砖国家人均 GDP（近 15 年），shape: {gdp_long.shape}')
gdp_long.tail(6)


In [ ]:
# ── 4.3  可视化：金砖国家人均 GDP 走势 ─────────────────────────────────
country_names = {'CN': '中国', 'IN': '印度', 'BR': '巴西',
                 'RU': '俄罗斯', 'ZA': '南非'}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue', 'darkorange', 'green', 'red', 'purple']

for code, name in country_names.items():
    # wbgapi labels=True 时用的是英文国家名，这里简化处理
    col = [c for c in gdp_long.columns if code in c or name in c]
    if col:
        ax.plot(gdp_long['year'], gdp_long[col[0]],
                label=name, linewidth=1.8)

ax.set_title('金砖国家人均 GDP 走势（现价美元）', fontsize=12)
ax.set_xlabel('年份')
ax.set_ylabel('人均 GDP（美元）')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch03_brics_gdp.png', dpi=150, bbox_inches='tight')
plt.show()


::: {.callout-tip}
## 提示词示范（World Bank 数据获取）

````
我需要用 wbgapi 获取以下数据：
1. 2000–2023 年 G20 国家的政府债务占 GDP 比（GC.DOD.TOTL.GD.ZS）
2. 转换为长表格式（列：country_code, country_name, year, debt_gdp_ratio）
3. 找出 2023 年债务率最高的 5 个国家
4. 绘制这 5 个国家自 2000 年以来的债务率走势折线图
````
:::


---

## 5　yfinance：国际市场数据

`yfinance` 通过 Yahoo Finance 获取全球股票、ETF、指数、外汇数据，免费无需 Key。

::: {.callout-warning}
## 大陆网络稳定性问题

在中国大陆，Yahoo Finance 的访问经常不稳定，
部分地区需要科学上网才能稳定获取长时间序列数据。

**替代方案**：
- 美国市场数据：使用 `fredapi`（指数、利率等宏观数据）
- 港股数据：使用 `akshare`（`ak.stock_hk_daily()`）
- 全球 ETF：先尝试 yfinance，不行时用 `pandas_datareader`
:::


In [ ]:
# ── 5.1  yfinance 使用示例（需要网络连接）────────────────────────────────
try:
    import yfinance as yf
    print(f'yfinance {yf.__version__} ✓')
except ImportError:
    print('请先安装：pip install yfinance')
    raise

# ── 获取标普 500 日度数据 ────────────────────────────────────────────────
try:
    sp500 = yf.download('^GSPC', start='2022-01-01', end='2024-12-31',
                        progress=False)
    sp500 = sp500['Close'].rename('sp500').reset_index()
    sp500.columns = ['date', 'sp500']
    sp500['ret'] = np.log(sp500['sp500'] / sp500['sp500'].shift(1))

    print(f'标普 500 数据：{len(sp500)} 行')
    print(sp500.tail(4).to_string(index=False))
    sp500.to_csv(f'{DATA_RAW}/sp500_daily.csv', index=False)

except Exception as e:
    print(f'yfinance 获取失败（可能是网络问题）：{e}')
    print('提示：如在大陆网络环境，建议使用科学上网后重试。')
    print('      或改用 fredapi 获取标普 500 数据：series_id = "SP500"')


---

## 6　数据存储方案

获取到数据后，选择合适的存储格式很重要。
三种最常用的格式，各有适用场景：

| 格式 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| **CSV** | 通用，任何工具都能读 | 大文件慢，不保留数据类型 | 小型数据集（< 100MB）、需要和他人共享 |
| **Parquet** | 列式压缩，读写快，保留类型 | 不能直接用文本编辑器查看 | 大型数据集（> 100MB）、纯 Python 环境 |
| **SQLite** | 支持 SQL 查询，多表关联 | 不适合矩阵运算 | 多张关联表（如 BasicInf + FinRatio + 行情）|

::: {.callout-note}
## 本课程的推荐选择

- 单张表 + 数据量 < 50MB → **CSV**（方便检查和共享）
- 全市场数据或宽面板 → **Parquet**（速度差距明显）
- 需要跨多表 JOIN 的分析 → **SQLite**（`pandas` 可直接读写）
:::


In [ ]:
# ── 6.1  三种存储格式的速度对比 ──────────────────────────────────────────
import time
import os

# 构造一个中等大小的示例 DataFrame（50 万行）
rng = np.random.default_rng(42)
n   = 500_000
df_big = pd.DataFrame({
    'stkcd'  : rng.choice(['000001','600028','000063','600036'], n),
    'date'   : pd.date_range('2020-01-01', periods=n, freq='h'),
    'price'  : rng.lognormal(5, 0.3, n),
    'volume' : rng.integers(1000, 1_000_000, n),
    'ret'    : rng.normal(0, 0.01, n),
})
print(f'测试数据：{len(df_big):,} 行 × {df_big.shape[1]} 列\n')

results = []
tmp = 'data_temp'
os.makedirs(tmp, exist_ok=True)

for fmt, write_fn, read_fn, path in [
    ('CSV',
     lambda: df_big.to_csv(f'{tmp}/test.csv', index=False),
     lambda: pd.read_csv(f'{tmp}/test.csv'),
     f'{tmp}/test.csv'),
    ('Parquet',
     lambda: df_big.to_parquet(f'{tmp}/test.parquet', index=False),
     lambda: pd.read_parquet(f'{tmp}/test.parquet'),
     f'{tmp}/test.parquet'),
]:
    # 写入
    t0 = time.time()
    write_fn()
    write_t = time.time() - t0

    # 读取
    t0 = time.time()
    _ = read_fn()
    read_t = time.time() - t0

    size_mb = os.path.getsize(path) / 1024**2
    results.append({'格式': fmt, '写入(s)': round(write_t,2),
                    '读取(s)': round(read_t,2), '文件大小(MB)': round(size_mb,1)})

print(pd.DataFrame(results).to_string(index=False))


In [ ]:
# ── 6.2  SQLite：多表存储与 SQL 查询 ─────────────────────────────────────
import sqlite3

DB_PATH = 'data_clean/finance.db'
os.makedirs('data_clean', exist_ok=True)

# ── 写入示例：把 hs300 存入 SQLite ──────────────────────────────────────
# 这里用前面获取的 hs300 数据演示（如果没有则跳过）
try:
    hs300_local = pd.read_csv(f'{DATA_RAW}/hs300_daily.csv')
    conn = sqlite3.connect(DB_PATH)
    hs300_local.to_sql('hs300_daily', conn, if_exists='replace', index=False)
    conn.close()
    print(f'hs300_daily 已存入 {DB_PATH}')
except FileNotFoundError:
    print('跳过（hs300_daily.csv 未找到，请先运行第 1 节的数据获取代码）')

# ── 用 pandas 直接读取 SQLite ────────────────────────────────────────────
try:
    conn   = sqlite3.connect(DB_PATH)
    result = pd.read_sql(
        'SELECT date, close, ret FROM hs300_daily ORDER BY date DESC LIMIT 10',
        conn
    )
    conn.close()
    print('\nSQL 查询结果（最新 10 行）：')
    print(result.to_string(index=False))
except Exception as e:
    print(f'SQLite 查询跳过：{e}')


::: {.callout-tip}
## 提示词示范（数据存储选择）

````
我有以下几种数据需要存储，请帮我推荐合适的格式并给出代码：

1. 沪深全市场（约 5000 只股票）的日度行情，10 年，约 1800 万行
   → 每周更新一次

2. 上市公司财务报表，约 5000 家公司 × 10 年，每年 4 次财报
   → 数据量约 20 万行，多张表（资产负债、利润表、现金流量表）

3. 临时的分析结果 DataFrame，约 10 万行，只在当前分析会话中使用

对每种情况，给出：存储格式推荐理由 + 读写代码 + 查询代码。
````
:::


---

## 7　综合案例：完成第一次个人作业

**作业要求**：用 AKShare 获取沪深 300 近三年日度收益率，
同时用 FRED API 获取同期联邦基金利率，
计算两者的 60 日滚动相关性，绘制双轴时序图。

注意：运行前请先完成沪深 300 数据获取（第 1 节）和 FRED Key 配置（第 3 节）。
如果没有 FRED Key，下面的 FRED 部分用模拟数据替代。


In [ ]:
# ── 7.1  读取已保存的沪深 300 数据 ──────────────────────────────────────
try:
    hs300 = pd.read_csv(f'{DATA_RAW}/hs300_daily.csv',
                        parse_dates=['date'])
    print(f'沪深 300 数据已加载：{len(hs300)} 行')
except FileNotFoundError:
    # 如果没有本地数据，用模拟数据替代（演示用）
    print('未找到本地数据，使用模拟数据演示...')
    rng   = np.random.default_rng(2024)
    dates = pd.date_range('2022-01-01', '2024-12-31', freq='B')
    hs300 = pd.DataFrame({
        'date' : dates,
        'close': 4000 * np.cumprod(1 + rng.normal(0.0003, 0.012, len(dates))),
    })
    hs300['ret'] = np.log(hs300['close'] / hs300['close'].shift(1))
    hs300 = hs300.dropna()
    print(f'模拟数据已生成：{len(hs300)} 行')


In [ ]:
# ── 7.2  联邦基金利率（或模拟数据）──────────────────────────────────────
try:
    fedfunds = pd.read_csv(f'{DATA_RAW}/fred_fedfunds.csv',
                           parse_dates=['date'])
    # FRED 是月度数据，需要插值到日度（前向填充）
    fedfunds = (fedfunds
                .set_index('date')
                .reindex(hs300['date'])
                .ffill()
                .reset_index())
    fedfunds.columns = ['date', 'fed_rate']
    print(f'FRED 联邦基金利率已加载并插值到日度')
except FileNotFoundError:
    print('未找到 FRED 数据，使用模拟利率...')
    # 模拟 2022–2024 的加息周期（0% → 5.5%）
    n = len(hs300)
    fedfunds = pd.DataFrame({
        'date'    : hs300['date'].values,
        'fed_rate': np.linspace(0.08, 5.33, n),
    })


In [ ]:
# ── 7.3  合并 + 计算 60 日滚动相关性 ────────────────────────────────────
merged = hs300[['date','ret']].merge(fedfunds, on='date', how='inner')

# 联邦基金利率转为日度变化（月度水平值 → 日度一阶差分）
merged['fed_chg'] = merged['fed_rate'].diff()

# 60 日滚动相关性（沪深 300 日收益率 vs 联邦基金利率日度变化）
merged['rolling_corr'] = (
    merged['ret'].rolling(60).corr(merged['fed_chg'])
)

merged = merged.dropna()
print(f'合并后有效数据：{len(merged)} 行')
print(f'滚动相关性均值：{merged["rolling_corr"].mean():.4f}')
print(f'滚动相关性标准差：{merged["rolling_corr"].std():.4f}')


In [ ]:
# ── 7.4  双轴时序图 ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})

# ── 上图：沪深 300 指数与联邦基金利率 ──────────────────────────────────
color_hs  = 'steelblue'
color_fed = 'darkorange'

ax1.plot(merged['date'], merged['ret'].cumsum() * 100,
         color=color_hs, linewidth=1.2, label='沪深 300 累计收益率（%，左轴）')
ax1.set_ylabel('累计收益率（%）', color=color_hs, fontsize=10)
ax1.tick_params(axis='y', labelcolor=color_hs)

ax1b = ax1.twinx()
ax1b.plot(merged['date'], merged['fed_rate'],
          color=color_fed, linewidth=1.5, linestyle='--',
          label='联邦基金利率（%，右轴）')
ax1b.set_ylabel('联邦基金利率（%）', color=color_fed, fontsize=10)
ax1b.tick_params(axis='y', labelcolor=color_fed)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
ax1.set_title('沪深 300 与美联储利率：走势与滚动相关性', fontsize=12)

# ── 下图：60 日滚动相关性 ────────────────────────────────────────────────
ax2.plot(merged['date'], merged['rolling_corr'],
         color='purple', linewidth=1.2)
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax2.fill_between(merged['date'], merged['rolling_corr'], 0,
                 where=merged['rolling_corr'] > 0,
                 alpha=0.2, color='green', label='正相关')
ax2.fill_between(merged['date'], merged['rolling_corr'], 0,
                 where=merged['rolling_corr'] < 0,
                 alpha=0.2, color='red', label='负相关')
ax2.set_ylabel('60 日滚动相关性', fontsize=10)
ax2.set_xlabel('日期', fontsize=10)
ax2.legend(loc='upper right', fontsize=9)
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch03_hs300_fed_corr.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存至 output/ch03_hs300_fed_corr.png')


---

## 8　章末练习（即第一次个人作业）

**练习 1（必做）**
完整运行第 7 节的综合案例代码，
用真实的 AKShare 数据（非模拟）和你申请的 FRED Key，
生成双轴时序图，保存到 `output/`，Push 到 GitHub 仓库。

**练习 2（必做）**
在第 7 节的基础上，额外分析：
① 用 `scipy.stats.pearsonr` 计算全样本期间的相关系数和 p 值
② 将数据分为「加息周期（fed_rate 上升）」和「降息/稳定期」两段，
   分别计算相关系数，比较是否有差异
③ 写 2–3 句话解释你观察到的规律

**练习 3（进阶）**
用 `wbgapi` 额外获取中国 GDP 同比增速（`NY.GDP.MKTP.KD.ZG`），
与沪深 300 年化收益率一起做折线图，
观察「经济基本面 vs 股市表现」的关系，写出你的观察（3–5 句话）。

**提交方式**：把所有代码（`.ipynb`）、图表（`output/`）和结论
推送到你的 GitHub 仓库，把仓库链接提交到课程系统。
